**Import libraries**

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

**Define constants**

In [ ]:
SOURCE_SILVER_TABLE = "dbo.sales_silver3"

DIM_CUSTOMER_TABLE = "dbo.dimcustomer_gold2"
DIM_PRODUCT_TABLE  = "dbo.dimproduct_gold2"
DIM_DATE_TABLE     = "dbo.dimdate_gold2"
FACT_SALES_TABLE   = "dbo.factsales_gold2"

NOTEBOOK_NAME = "GoldNotebook"
HIGH_DATE_STR = "9999-12-31 23:59:59"


**READ SILVER SOURCE**

In [ ]:
sales_silver_df = spark.read.table(SOURCE_SILVER_TABLE)

print("sales_silver count:", sales_silver_df.count())

**CREATE GOLD TABLES IF NEEDED**

In [ ]:
DeltaTable.createIfNotExists(spark) \
    .tableName(DIM_CUSTOMER_TABLE) \
    .addColumn("CustomerKey", StringType()) \
    .addColumn("CustomerBK", StringType()) \
    .addColumn("CustomerName", StringType()) \
    .addColumn("Email", StringType()) \
    .addColumn("EffectiveStartDate", TimestampType()) \
    .addColumn("EffectiveEndDate", TimestampType()) \
    .addColumn("IsCurrent", BooleanType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(DIM_PRODUCT_TABLE) \
    .addColumn("ItemKey", StringType()) \
    .addColumn("ItemBK", StringType()) \
    .addColumn("Item", StringType()) \
    .addColumn("EffectiveStartDate", TimestampType()) \
    .addColumn("EffectiveEndDate", TimestampType()) \
    .addColumn("IsCurrent", BooleanType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(DIM_DATE_TABLE) \
    .addColumn("DateKey", IntegerType()) \
    .addColumn("FullDate", DateType()) \
    .addColumn("Day", IntegerType()) \
    .addColumn("Month", IntegerType()) \
    .addColumn("MonthName", StringType()) \
    .addColumn("Quarter", IntegerType()) \
    .addColumn("Year", IntegerType()) \
    .addColumn("WeekOfYear", IntegerType()) \
    .addColumn("DayOfWeek", IntegerType()) \
    .addColumn("DayName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .execute()

DeltaTable.createIfNotExists(spark) \
    .tableName(FACT_SALES_TABLE) \
    .addColumn("SalesKey", StringType()) \
    .addColumn("SalesOrderNumber", StringType()) \
    .addColumn("SalesOrderLineNumber", StringType()) \
    .addColumn("DateKey", IntegerType()) \
    .addColumn("OrderDate", DateType()) \
    .addColumn("CustomerKey", StringType()) \
    .addColumn("ItemKey", StringType()) \
    .addColumn("Quantity", IntegerType()) \
    .addColumn("UnitPrice", DoubleType()) \
    .addColumn("Tax", DoubleType()) \
    .addColumn("IsFlagged", BooleanType()) \
    .addColumn("SourceFileName", StringType()) \
    .addColumn("LoadTimestamp", TimestampType()) \
    .addColumn("CreatedTS", TimestampType()) \
    .addColumn("ModifiedTS", TimestampType()) \
    .addColumn("CreatedBy", StringType()) \
    .addColumn("ModifiedBy", StringType()) \
    .execute()

**BUILD / LOAD DIMDATE**

In [ ]:
date_source_df = (
    sales_silver_df
    .select(to_date(col("OrderDate")).alias("FullDate"))
    .filter(col("FullDate").isNotNull())
    .dropDuplicates(["FullDate"])
    .withColumn("DateKey", date_format(col("FullDate"), "yyyyMMdd").cast("int"))
    .withColumn("Day", dayofmonth(col("FullDate")))
    .withColumn("Month", month(col("FullDate")))
    .withColumn("MonthName", date_format(col("FullDate"), "MMMM"))
    .withColumn("Quarter", quarter(col("FullDate")))
    .withColumn("Year", year(col("FullDate")))
    .withColumn("WeekOfYear", weekofyear(col("FullDate")))
    .withColumn("DayOfWeek", dayofweek(col("FullDate")))
    .withColumn("DayName", date_format(col("FullDate"), "EEEE"))
    .withColumn("LoadTimestamp", current_timestamp())
    .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
)
dim_date_delta = DeltaTable.forName(spark, DIM_DATE_TABLE)

dim_date_delta.alias("tgt").merge(
    date_source_df.alias("src"),
    "tgt.DateKey = src.DateKey"
).whenNotMatchedInsert(values={
    "DateKey": "src.DateKey",
    "FullDate": "src.FullDate",
    "Day": "src.Day",
    "Month": "src.Month",
    "MonthName": "src.MonthName",
    "Quarter": "src.Quarter",
    "Year": "src.Year",
    "WeekOfYear": "src.WeekOfYear",
    "DayOfWeek": "src.DayOfWeek",
    "DayName": "src.DayName",
    "LoadTimestamp": "src.LoadTimestamp",
    "CreatedBy": "src.CreatedBy"
}).execute()

**DIMCUSTOMER - PREPARE SOURCE**

In [ ]:
customer_src = (
    sales_silver_df
    .select(
        trim(col("Email")).alias("CustomerBK"),
        trim(col("CustomerName")).alias("CustomerName"),
        trim(col("Email")).alias("Email"),
        col("FileName").alias("SourceFileName"),
        col("CreatedTS"),
        col("ModifiedTS")
    )
    .filter(col("CustomerBK").isNotNull() & (col("CustomerBK") != ""))
    .withColumn(
        "EffectiveStartDate",
        coalesce(col("ModifiedTS"), col("CreatedTS"), current_timestamp())
    )
    .dropDuplicates(["CustomerBK", "CustomerName", "Email", "EffectiveStartDate"])
)

customer_dim = spark.read.table(DIM_CUSTOMER_TABLE)
is_first_customer_load = customer_dim.count() == 0

print("customer_src count:", customer_src.count())
print("Is first customer load?", is_first_customer_load)

**DIMCUSTOMER - FIRST LOAD OR INCREMENTAL SCD2**

In [ ]:
if is_first_customer_load:
    customer_versions = (
        customer_src
        .withColumn(
            "VersionHash",
            sha2(
                concat_ws(
                    "|",
                    coalesce(col("CustomerName"), lit("")),
                    coalesce(col("Email"), lit(""))
                ),
                256
            )
        )
    )

    same_version_window = Window.partitionBy("CustomerBK", "VersionHash").orderBy("EffectiveStartDate")

    customer_versions = (
        customer_versions
        .withColumn("rn_same_version", row_number().over(same_version_window))
        .filter(col("rn_same_version") == 1)
        .drop("rn_same_version")
    )

    scd_window = Window.partitionBy("CustomerBK").orderBy("EffectiveStartDate")

    customer_scd2 = (
        customer_versions
        .withColumn("NextStartDate", lead("EffectiveStartDate").over(scd_window))
        .withColumn(
            "EffectiveEndDate",
            when(col("NextStartDate").isNull(), to_timestamp(lit(HIGH_DATE_STR)))
            .otherwise(col("NextStartDate"))
        )
        .withColumn(
            "IsCurrent",
            when(col("NextStartDate").isNull(), lit(True)).otherwise(lit(False))
        )
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .withColumn(
            "CustomerKey",
            sha2(
                concat_ws(
                    "|",
                    col("CustomerBK"),
                    date_format(col("EffectiveStartDate"), "yyyy-MM-dd HH:mm:ss.SSS")
                ),
                256
            )
        )
        .select(
            "CustomerKey",
            "CustomerBK",
            "CustomerName",
            "Email",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "SourceFileName",
            "LoadTimestamp",
            "CreatedTS",
            "ModifiedTS",
            "CreatedBy",
            "ModifiedBy"
        )
    )

    customer_scd2.write.format("delta").mode("append").saveAsTable(DIM_CUSTOMER_TABLE)
    print("Inserted first-load customer SCD2 rows.")

else:
    customer_current = customer_dim.filter(col("IsCurrent") == True)

    customer_changes = (
        customer_src.alias("src")
        .join(
            customer_current.alias("tgt"),
            col("src.CustomerBK") == col("tgt.CustomerBK"),
            "inner"
        )
        .filter(
            (coalesce(col("src.CustomerName"), lit("")) != coalesce(col("tgt.CustomerName"), lit(""))) |
            (coalesce(col("src.Email"), lit("")) != coalesce(col("tgt.Email"), lit("")))
        )
        .select(
            col("src.CustomerBK"),
            col("src.CustomerName"),
            col("src.Email"),
            col("src.SourceFileName"),
            col("src.CreatedTS"),
            col("src.ModifiedTS"),
            col("src.EffectiveStartDate")
        )
    )
    
    customer_new_bk = (
        customer_src.alias("src")
        .join(
            customer_current.alias("tgt"),
            col("src.CustomerBK") == col("tgt.CustomerBK"),
            "left_anti"
        )
    )

    delta_customer = DeltaTable.forName(spark, DIM_CUSTOMER_TABLE)

    delta_customer.alias("tgt").merge(
        customer_changes.alias("src"),
        "tgt.CustomerBK = src.CustomerBK AND tgt.IsCurrent = true"
    ).whenMatchedUpdate(set={
        "EffectiveEndDate": "src.EffectiveStartDate",
        "IsCurrent": "false",
        "LoadTimestamp": "current_timestamp()",
        "ModifiedBy": f"'{NOTEBOOK_NAME}'"
    }).execute()

    customer_inserts = (
        customer_new_bk.unionByName(customer_changes, allowMissingColumns=True)
        .dropDuplicates(["CustomerBK", "CustomerName", "Email", "EffectiveStartDate"])
        .withColumn("EffectiveStartDate", coalesce(col("EffectiveStartDate"), current_timestamp()))
        .withColumn("EffectiveEndDate", to_timestamp(lit(HIGH_DATE_STR)))
        .withColumn("IsCurrent", lit(True))
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .withColumn(
            "CustomerKey",
            sha2(
                concat_ws(
                    "|",
                    col("CustomerBK"),
                    date_format(col("EffectiveStartDate"), "yyyy-MM-dd HH:mm:ss.SSS")
                ),
                256
            )
        )
        .select(
            "CustomerKey",
            "CustomerBK",
            "CustomerName",
            "Email",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "SourceFileName",
            "LoadTimestamp",
            "CreatedTS",
            "ModifiedTS",
            "CreatedBy",
            "ModifiedBy"
        )
    )

    customer_inserts.write.format("delta").mode("append").saveAsTable(DIM_CUSTOMER_TABLE)
    print("Inserted incremental customer rows.")

**DIMPRODUCT - PREPARE SOURCE**

In [ ]:
product_src = (
    sales_silver_df
    .select(
        trim(col("Item")).alias("ItemBK"),
        trim(col("Item")).alias("Item"),
        col("FileName").alias("SourceFileName"),
        col("CreatedTS"),
        col("ModifiedTS")
    )
    .filter(col("ItemBK").isNotNull() & (col("ItemBK") != ""))
    .withColumn(
        "EffectiveStartDate",
        coalesce(col("ModifiedTS"), col("CreatedTS"), current_timestamp())
    )
    .dropDuplicates(["ItemBK", "Item", "EffectiveStartDate"])
)

product_dim = spark.read.table(DIM_PRODUCT_TABLE)
is_first_product_load = product_dim.count() == 0

print("product_src count:", product_src.count())
print("Is first product load?", is_first_product_load)

**DIMPRODUCT - FIRST LOAD OR INCREMENTAL SCD2**

In [ ]:
if is_first_product_load:
    product_versions = (
        product_src
        .withColumn(
            "VersionHash",
            sha2(concat_ws("|", coalesce(col("Item"), lit(""))), 256)
        )
    )

    same_version_window = Window.partitionBy("ItemBK", "VersionHash").orderBy("EffectiveStartDate")

    product_versions = (
        product_versions
        .withColumn("rn_same_version", row_number().over(same_version_window))
        .filter(col("rn_same_version") == 1)
        .drop("rn_same_version")
    )

    scd_window = Window.partitionBy("ItemBK").orderBy("EffectiveStartDate")

    product_scd2 = (
        product_versions
        .withColumn("NextStartDate", lead("EffectiveStartDate").over(scd_window))
        .withColumn(
            "EffectiveEndDate",
            when(col("NextStartDate").isNull(), to_timestamp(lit(HIGH_DATE_STR)))
            .otherwise(col("NextStartDate"))
        )
        .withColumn(
            "IsCurrent",
            when(col("NextStartDate").isNull(), lit(True)).otherwise(lit(False))
        )
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .withColumn(
            "ItemKey",
            sha2(
                concat_ws(
                    "|",
                    col("ItemBK"),
                    date_format(col("EffectiveStartDate"), "yyyy-MM-dd HH:mm:ss.SSS")
                ),
                256
            )
        )
        .select(
            "ItemKey",
            "ItemBK",
            "Item",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "SourceFileName",
            "LoadTimestamp",
            "CreatedTS",
            "ModifiedTS",
            "CreatedBy",
            "ModifiedBy"
        )
    )

    product_scd2.write.format("delta").mode("append").saveAsTable(DIM_PRODUCT_TABLE)
    print("Inserted first-load product SCD2 rows.")

else:
    product_current = product_dim.filter(col("IsCurrent") == True)

    product_changes = (
        product_src.alias("src")
        .join(
            product_current.alias("tgt"),
            col("src.ItemBK") == col("tgt.ItemBK"),
            "inner"
        )
        .filter(
            coalesce(col("src.Item"), lit("")) != coalesce(col("tgt.Item"), lit(""))
        )
        .select(
            col("src.ItemBK"),
            col("src.Item"),
            col("src.SourceFileName"),
            col("src.CreatedTS"),
            col("src.ModifiedTS"),
            col("src.EffectiveStartDate")
        )
    )
    product_new_bk = (
        product_src.alias("src")
        .join(
            product_current.alias("tgt"),
            col("src.ItemBK") == col("tgt.ItemBK"),
            "left_anti"
        )
    )

    delta_product = DeltaTable.forName(spark, DIM_PRODUCT_TABLE)

    delta_product.alias("tgt").merge(
        product_changes.alias("src"),
        "tgt.ItemBK = src.ItemBK AND tgt.IsCurrent = true"
    ).whenMatchedUpdate(set={
        "EffectiveEndDate": "src.EffectiveStartDate",
        "IsCurrent": "false",
        "LoadTimestamp": "current_timestamp()",
        "ModifiedBy": f"'{NOTEBOOK_NAME}'"
    }).execute()

    product_inserts = (
        product_new_bk.unionByName(product_changes, allowMissingColumns=True)
        .dropDuplicates(["ItemBK", "Item", "EffectiveStartDate"])
        .withColumn("EffectiveStartDate", coalesce(col("EffectiveStartDate"), current_timestamp()))
        .withColumn("EffectiveEndDate", to_timestamp(lit(HIGH_DATE_STR)))
        .withColumn("IsCurrent", lit(True))
        .withColumn("LoadTimestamp", current_timestamp())
        .withColumn("CreatedBy", lit(NOTEBOOK_NAME))
        .withColumn("ModifiedBy", lit(NOTEBOOK_NAME))
        .withColumn(
            "ItemKey",
            sha2(
                concat_ws(
                    "|",
                    col("ItemBK"),
                    date_format(col("EffectiveStartDate"), "yyyy-MM-dd HH:mm:ss.SSS")
                ),
                256
            )
        )
        .select(
            "ItemKey",
            "ItemBK",
            "Item",
            "EffectiveStartDate",
            "EffectiveEndDate",
            "IsCurrent",
            "SourceFileName",
            "LoadTimestamp",
            "CreatedTS",
            "ModifiedTS",
            "CreatedBy",
            "ModifiedBy"
        )
    )

    product_inserts.write.format("delta").mode("append").saveAsTable(DIM_PRODUCT_TABLE)
    print("Inserted incremental product rows.")

**VALIDATE CURRENT-ROW UNIQUENESS**

In [ ]:
customer_current_df = (
    spark.read.table(DIM_CUSTOMER_TABLE)
    .filter(col("IsCurrent") == True)
    .withColumn(
        "rn",
        row_number().over(
            Window.partitionBy("CustomerBK").orderBy(col("EffectiveStartDate").desc())
        )
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

product_current_df = (
    spark.read.table(DIM_PRODUCT_TABLE)
    .filter(col("IsCurrent") == True)
    .withColumn(
        "rn",
        row_number().over(
            Window.partitionBy("ItemBK").orderBy(col("EffectiveStartDate").desc())
        )
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

date_dim_df = spark.read.table(DIM_DATE_TABLE)

**BUILD FACT SOURCE**
# Fact grain = SalesOrderNumber + SalesOrderLineNumber

In [ ]:
fact_source_df = (
    sales_silver_df.alias("s")
    .join(
        customer_current_df.alias("c"),
        trim(col("s.Email")) == col("c.CustomerBK"),
        "left"
    )
    .join(
        product_current_df.alias("p"),
        trim(col("s.Item")) == col("p.ItemBK"),
        "left"
    )
    .join(
        date_dim_df.alias("d"),
        to_date(col("s.OrderDate")) == col("d.FullDate"),
        "left"
    )
    .select(
        sha2(
            concat_ws(
                "|",
                trim(col("s.SalesOrderNumber")),
                trim(col("s.SalesOrderLineNumber"))
            ),
            256
        ).alias("SalesKey"),
        trim(col("s.SalesOrderNumber")).alias("SalesOrderNumber"),
        trim(col("s.SalesOrderLineNumber")).alias("SalesOrderLineNumber"),
        col("d.DateKey"),
        to_date(col("s.OrderDate")).alias("OrderDate"),
        col("c.CustomerKey"),
        col("p.ItemKey"),
        col("s.Quantity").cast("int").alias("Quantity"),
        col("s.UnitPrice").cast("double").alias("UnitPrice"),
        col("s.Tax").cast("double").alias("Tax"),
        col("s.IsFlagged").cast("boolean").alias("IsFlagged"),
        col("s.FileName").alias("SourceFileName"),
        current_timestamp().alias("LoadTimestamp"),
        col("s.CreatedTS"),
        col("s.ModifiedTS"),
        lit(NOTEBOOK_NAME).alias("CreatedBy"),
        lit(NOTEBOOK_NAME).alias("ModifiedBy")
    )
)

print("fact_source count:", fact_source_df.count())

**LOAD FACTSALES**

In [ ]:
fact_sales_delta = DeltaTable.forName(spark, FACT_SALES_TABLE)

fact_sales_delta.alias("tgt").merge(
    fact_source_df.alias("src"),
    """
    tgt.SalesOrderNumber = src.SalesOrderNumber
    AND tgt.SalesOrderLineNumber = src.SalesOrderLineNumber
    """
).whenMatchedUpdate(set={
    "DateKey": "src.DateKey",
    "OrderDate": "src.OrderDate",
    "CustomerKey": "src.CustomerKey",
    "ItemKey": "src.ItemKey",
    "Quantity": "src.Quantity",
    "UnitPrice": "src.UnitPrice",
    "Tax": "src.Tax",
    "IsFlagged": "src.IsFlagged",
    "SourceFileName": "src.SourceFileName",
    "LoadTimestamp": "src.LoadTimestamp",
    "CreatedTS": "src.CreatedTS",
    "ModifiedTS": "src.ModifiedTS",
    "ModifiedBy": "src.ModifiedBy"
}).whenNotMatchedInsert(values={
    "SalesKey": "src.SalesKey",
    "SalesOrderNumber": "src.SalesOrderNumber",
    "SalesOrderLineNumber": "src.SalesOrderLineNumber",
    "DateKey": "src.DateKey",
    "OrderDate": "src.OrderDate",
    "CustomerKey": "src.CustomerKey",
    "ItemKey": "src.ItemKey",
    "Quantity": "src.Quantity",
    "UnitPrice": "src.UnitPrice",
    "Tax": "src.Tax",
    "IsFlagged": "src.IsFlagged",
    "SourceFileName": "src.SourceFileName",
    "LoadTimestamp": "src.LoadTimestamp",
    "CreatedTS": "src.CreatedTS",
    "ModifiedTS": "src.ModifiedTS",
    "CreatedBy": "src.CreatedBy",
    "ModifiedBy": "src.ModifiedBy"
}).execute()

**FINAL VALIDATION**

In [ ]:
print("dimdate count:", spark.read.table(DIM_DATE_TABLE).count())
print("dimcustomer count:", spark.read.table(DIM_CUSTOMER_TABLE).count())
print("dimproduct count:", spark.read.table(DIM_PRODUCT_TABLE).count())
print("factsales count:", spark.read.table(FACT_SALES_TABLE).count())

print(
    "distinct silver order-line count:",
    sales_silver_df.select("SalesOrderNumber", "SalesOrderLineNumber").distinct().count()
)